# Welcome!

This example shows a very basic usage case of MultiBench. In particular, it demonstrates how to use MultiBench with the affective computing dataset MOSI, and how to use it with a very simple fusion model. 

While this will be simple, it will show off most of the capabilities of MultiBench, and most of the conventions at the heart of the system.

To begin, let's set up the interpreter to resolve MultiBench imports correctly.

In [1]:
import sys
import os

# Add the MultiBench root directory to the path so imports resolve correctly
repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"Using MultiBench from: {repo_root}")

Using MultiBench from: /home/bagus/github/multibench


The cell below verifies the MOSI data file exists at `data/affect/mosi_raw.pkl` inside the repo. If the path is wrong, update `data_path` accordingly.

In [2]:
data_path = os.path.join(repo_root, 'data', 'affect', 'mosi_raw.pkl')
print(f"Data path: {data_path}")
print(f"Data file exists: {os.path.exists(data_path)}")

Data path: /home/bagus/github/multibench/data/affect/mosi_raw.pkl
Data file exists: True


Make sure all required dependencies are installed (see the repo's `README.md` for the full list). For a quick install: `pip install torch torchvision torchaudio`.

From here, let's import some of MultiBench and get working:

In [3]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


**Note about torchtext compatibility:**

This notebook uses GloVe embeddings for text processing. Due to compatibility issues with newer versions of PyTorch and torchtext, MultiBench now includes its own GloVe loader (`datasets/affect/glove_loader.py`) that automatically replaces torchtext when needed.

The first time you run the data loading, it will download the GloVe embeddings (approximately 2GB for the 840B corpus). These will be cached in `~/.cache/glove/` for future use.

If you encounter issues with torchtext, the system will automatically fall back to the built-in GloVe loader, so no action is required on your part.

First, we'll import and create the dataloader for the MOSI dataset, which we're working with:

In [4]:
# Import the associated dataloader for affect datasets, which MOSI is a part of.
from datasets.affect.get_data import get_dataloader

# Create the training, validation, and test-set dataloaders. 
traindata, validdata, testdata = get_dataloader(
    data_path, robust_test=False, max_pad=True, data_type='mosi', max_seq_len=50)

Then, let's define our MultiModal model to test. MultiBench divides models into three separate portions.

Firstly, let's define the encoders of the raw modality information, which come from the "unimodals" section of MultiBench:

In [5]:
# Here, we'll import several common modules should you want to mess with this more.
from unimodals.common_models import GRU, MLP, Sequential, Identity 

# As this example is meant to be simple and easy to train, we'll pass in identity
# functions for each of the modalities in MOSI:
encoders = [Identity().to(device), Identity().to(device), Identity().to(device)]

Then, let's define the fusion paradigm, which will govern how we take the current modalities, and combine them.

For this example, we'll use the ConcatEarly fusion, which just concatenates the inputs along the second dimension.

In [6]:
# Import a fusion paradigm, in this case early concatenation.
from fusions.common_fusions import ConcatEarly  # noqa

# Initialize the fusion module
fusion = ConcatEarly().to(device)

Lastly, we'll define a 'head' module, which takes the output of the fusion module, and applies transformations to get an output that correponds to our problem - sarcasm detection.

In [7]:
head = Sequential(GRU(409, 512, dropout=True, has_padding=False,
                  batch_first=True, last_only=True), MLP(512, 512, 1)).to(device)

And with that, we're almost done! Now we just need to put them into one of MultiBench's training loops, and set it running:

In [ ]:
# Standard supervised learning training loop
from training_structures.Supervised_Learning import train, test

# For more information regarding parameters for any system, feel free to check out the documentation
# at multibench.readthedocs.io!
train(encoders, fusion, head, traindata, validdata, 100, task="regression", optimtype=torch.optim.AdamW,
      is_packed=False, lr=1e-3, save='mosi_ef_r0.pt', weight_decay=0.01, objective=torch.nn.L1Loss())

print("Testing:")
model = torch.load('mosi_ef_r0.pt', map_location=device).to(device)
test(model, testdata, 'affect', is_packed=False,
     criterion=torch.nn.L1Loss(), task="posneg-classification", no_robust=True)

Epoch 0 train loss: 1.3318481058788225
Epoch 0 valid loss: 1.3968757424399116
Saving Best
Epoch 1 train loss: 1.32791341847028
Epoch 1 valid loss: 1.4028993822703852
Epoch 2 train loss: 1.3220721476262005
Epoch 2 valid loss: 1.386896352901637
Saving Best
Epoch 3 train loss: 1.3183917779524812
Epoch 3 valid loss: 1.392576986384169
Epoch 4 train loss: 1.3203976036790563
Epoch 4 valid loss: 1.3830485488766822
Saving Best
Epoch 5 train loss: 1.3187824160627155
Epoch 5 valid loss: 1.389201924065563
Epoch 6 train loss: 1.3145382362980444
Epoch 6 valid loss: 1.381537050844353
Saving Best
Epoch 7 train loss: 1.3184487457305123
Epoch 7 valid loss: 1.3898373006660247
Epoch 8 train loss: 1.3332633693889073
Epoch 8 valid loss: 1.3898504036609258
Epoch 9 train loss: 1.317722907962041
Epoch 9 valid loss: 1.3813474612815357
Saving Best
Epoch 10 train loss: 1.3162231732418272
Epoch 10 valid loss: 1.388633072933304
Epoch 11 train loss: 1.3179283064506246
Epoch 11 valid loss: 1.38245719392723
Epoch 12 t

And with that, you've taken your first step into using MultiBench! We hope you find the library useful, and feel free to make an issue on GitHub should there be any confusions regarding how to use an aspect of the package.